# 📗 Bài 2 — Gradient Descent

**Tuần 5** · ⭐ **Notebook quan trọng nhất Phase 02**

Bạn sắp tự code lại cơ chế mà **mọi** mô hình AI — từ hồi quy tuyến tính tới GPT — dùng để học.

Đừng chỉ chạy. Hãy **đổi số rồi chạy lại** và nhìn đồ thị thay đổi.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (11, 4)
rng = np.random.default_rng(42)

## 1. Ẩn dụ

> Bạn đứng trên sườn đồi trong sương mù dày đặc, muốn xuống thung lũng. Không nhìn được xa, nhưng **cảm nhận được độ dốc dưới chân**. Bạn bước một bước theo hướng dốc xuống nhất, rồi lặp lại.

| Ẩn dụ | Toán học |
|---|---|
| Độ cao chỗ đứng | **Loss** — model sai bao nhiêu |
| Vị trí của bạn | **Tham số** (weights) |
| Độ dốc dưới chân | **Gradient** |
| Độ dài mỗi bước | **Learning rate** |
| Đáy thung lũng | Loss nhỏ nhất |

## 2. Hàm loss để tập: $f(w) = (w-3)^2$

Đáy nằm tại $w = 3$. Ta *biết trước* đáp án — mục đích là xem thuật toán có tự tìm ra không.

In [ ]:
def loss(w):
    return (w - 3) ** 2

def gradient(w):
    return 2 * (w - 3)          # đạo hàm của (w-3)^2


w_range = np.linspace(-2, 8, 200)

plt.plot(w_range, loss(w_range), lw=2)
plt.axvline(3, color="green", ls="--", label="đáy: w = 3")
plt.xlabel("w"); plt.ylabel("loss"); plt.legend(); plt.grid(alpha=.3)
plt.title("Hàm loss f(w) = (w-3)²")
plt.show()

## 3. Gradient cho biết gì?

In [ ]:
for w in [0, 1, 2.5, 3, 3.5, 5, 8]:
    g = gradient(w)
    huong = "→ phải" if g < 0 else ("← trái" if g > 0 else "  ĐỨNG YÊN")
    print(f"  w = {w:4}   loss = {loss(w):6.2f}   gradient = {g:6.2f}   cần đi {huong}")

**Hai điều gradient nói với ta:**

| | Ý nghĩa |
|---|---|
| **Dấu** | Đang ở phía nào của đáy (dương = bên phải, âm = bên trái) |
| **Độ lớn** | Đang cách đáy bao xa (càng xa càng lớn) |

Công thức `w = w - lr * gradient` tự xử lý đúng cả hai chiều:
- gradient **dương** → trừ đi số dương → `w` giảm → đi sang trái ✓
- gradient **âm** → trừ đi số âm → `w` tăng → đi sang phải ✓

Đó là lý do có **dấu trừ** trong công thức.

Và vì gradient nhỏ dần khi tới gần đáy, thuật toán tự động "hạ cánh mềm" mà ta không phải làm gì thêm.

## 4. Vòng lặp gradient descent — 3 dòng làm nên deep learning

In [ ]:
def gradient_descent(w_bat_dau, lr, so_buoc):
    w = w_bat_dau
    lich_su = [w]
    for _ in range(so_buoc):
        w = w - lr * gradient(w)        # <- TOÀN BỘ DEEP LEARNING NẰM Ở ĐÂY
        lich_su.append(w)
    return lich_su


ls = gradient_descent(w_bat_dau=0.0, lr=0.1, so_buoc=30)

for i in [0, 1, 2, 3, 5, 10, 20, 30]:
    print(f"  bước {i:2}:  w = {ls[i]:8.5f}   loss = {loss(ls[i]):9.6f}")

### Nhìn nó "lăn xuống dốc"

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2)

ax1.plot(w_range, loss(w_range), lw=2, alpha=.5)
ax1.plot(ls, [loss(w) for w in ls], "ro-", ms=5, lw=1)
ax1.axvline(3, color="green", ls="--")
ax1.set_xlabel("w"); ax1.set_ylabel("loss"); ax1.set_title("Đường đi trên mặt loss")
ax1.grid(alpha=.3)

ax2.plot([loss(w) for w in ls], lw=2)
ax2.set_xlabel("bước"); ax2.set_ylabel("loss"); ax2.set_title("Loss giảm theo thời gian")
ax2.grid(alpha=.3)

plt.tight_layout(); plt.show()

Chú ý: các bước **thưa dần** khi tới gần đáy. Không phải ta lập trình như vậy — gradient nhỏ dần tự làm điều đó.

## 5. ⭐ Learning rate — siêu tham số quan trọng nhất

Đây là phần cần nhìn kỹ nhất cả notebook.

In [ ]:
cac_lr = [0.01, 0.1, 0.5, 0.9, 1.0, 1.05]
fig, axes = plt.subplots(2, 3, figsize=(14, 7))

for ax, lr in zip(axes.flat, cac_lr):
    ls = gradient_descent(0.0, lr, 25)
    ax.plot(w_range, loss(w_range), alpha=.35, lw=2)
    ax.plot(ls, [loss(w) for w in ls], "ro-", ms=4, lw=1)
    ax.axvline(3, color="green", ls="--", alpha=.6)
    ax.set_title(f"lr = {lr}   →  w cuối = {ls[-1]:.2f}", fontsize=10)
    ax.set_ylim(-2, 60)
    ax.set_xlim(-3, 9)
    ax.grid(alpha=.25)

plt.tight_layout(); plt.show()

In [ ]:
# Nhìn bằng số cho rõ
print(f"{'lr':>6} | {'w sau 25 bước':>16} | nhận xét")
print("-" * 58)
for lr in [0.001, 0.01, 0.1, 0.5, 0.9, 1.0, 1.05, 1.5]:
    w_cuoi = gradient_descent(0.0, lr, 25)[-1]
    if not np.isfinite(w_cuoi) or abs(w_cuoi) > 1e6:
        nx = "PHÂN KỲ — bay ra vô cực"
    elif abs(w_cuoi - 3) < 0.01:
        nx = "hội tụ tốt"
    elif abs(w_cuoi - 3) < 1:
        nx = "gần tới, cần thêm bước"
    elif abs(w_cuoi) > 100:
        nx = "PHÂN KỲ"
    else:
        nx = "quá chậm hoặc dao động"
    print(f"{lr:>6} | {w_cuoi:>16.4f} | {nx}")

| Learning rate | Hiện tượng |
|---|---|
| **Quá nhỏ** (0.001) | Học rất chậm, 25 bước vẫn còn xa đích |
| **Vừa** (0.1 – 0.5) | Hội tụ mượt ✓ |
| **Sát ngưỡng** (0.9 – 1.0) | Nhảy qua nhảy lại, không ổn định |
| **Vượt ngưỡng** (> 1.0) | **Phân kỳ** — bay ra vô cực |

> 💡 **Chẩn đoán bạn sẽ dùng suốt sự nghiệp:** thấy loss ra `NaN` hoặc tăng vọt → gần như chắc chắn learning rate quá lớn. Giảm 10 lần rồi chạy lại.

Với hàm này ngưỡng phân kỳ là `lr = 1.0`. Mỗi hàm có ngưỡng riêng và **ta không biết trước** — nên phải thử nghiệm. Đó là lý do learning rate là siêu tham số đầu tiên cần chỉnh.

In [ ]:
# 🔬 THỬ NGHIỆM: đổi hai số dưới đây rồi chạy lại
LR = 0.3
BAT_DAU = -5.0

ls = gradient_descent(BAT_DAU, LR, 40)
plt.plot(w_range, loss(w_range), alpha=.4, lw=2)
plt.plot(ls, [loss(w) for w in ls], "ro-", ms=4)
plt.axvline(3, color="green", ls="--")
plt.title(f"lr={LR}, bắt đầu từ w={BAT_DAU}  →  w cuối = {ls[-1]:.4f}")
plt.grid(alpha=.3); plt.show()

## 6. Local minimum

In [ ]:
def loss_ganh(w):
    return 0.05 * w**4 - 0.5 * w**3 + 0.2 * w**2 + 3 * w + 10

def grad_ganh(w):
    return 0.2 * w**3 - 1.5 * w**2 + 0.4 * w + 3


def gd_ganh(w0, lr, n):
    w, ls = w0, [w0]
    for _ in range(n):
        w = w - lr * grad_ganh(w)
        ls.append(w)
    return ls


wr = np.linspace(-4, 10, 300)
plt.plot(wr, loss_ganh(wr), lw=2, alpha=.5)

for w0, mau in [(-3.0, "red"), (6.0, "blue")]:
    ls = gd_ganh(w0, 0.02, 80)
    plt.plot(ls, [loss_ganh(w) for w in ls], "o-", color=mau, ms=3, lw=1,
             label=f"bắt đầu w={w0} → dừng ở {ls[-1]:.2f}")

plt.legend(); plt.grid(alpha=.3)
plt.title("Cùng thuật toán, xuất phát khác nhau → kết quả khác nhau")
plt.show()

Hai điểm xuất phát khác nhau rơi vào **hai hố khác nhau**. Một hố sâu hơn (*global minimum*), một hố cạn hơn (*local minimum*).

**Có đáng lo không?** Với mạng neural nhiều chiều thì **ít hơn bạn tưởng**: trong không gian hàng triệu chiều, để bị kẹt thì mọi chiều đều phải đi lên — xác suất cực thấp. Hầu hết điểm dừng là *saddle point* (yên ngựa), và các kỹ thuật hiện đại (momentum, Adam) vượt qua chúng khá tốt.

## 7. Áp dụng thật: hồi quy tuyến tính

Giờ dùng gradient descent để **học từ dữ liệu** thay vì tối thiểu hoá một hàm cho sẵn.

In [ ]:
# Sinh dữ liệu giả với tham số THẬT mà ta biết trước
W_THAT = np.array([2.0, -1.0])
B_THAT = 5.0

X = rng.normal(size=(200, 2))
y = X @ W_THAT + B_THAT + rng.normal(0, 0.3, size=200)     # thêm nhiễu

print("X.shape =", X.shape)
print("y.shape =", y.shape)
print(f"\nTham số THẬT: w = {W_THAT}, b = {B_THAT}")
print("Xem model có tự tìm lại được không...")

In [ ]:
def train(X, y, lr=0.1, so_epoch=300):
    n = len(X)
    w = np.zeros(X.shape[1])       # khởi tạo bằng 0
    b = 0.0
    lich_su = []

    for _ in range(so_epoch):
        y_pred = X @ w + b                       # 1. forward
        sai_so = y_pred - y
        lich_su.append(float(np.mean(sai_so**2)))  # 2. loss (MSE)

        grad_w = (2 / n) * (X.T @ sai_so)        # 3. gradient
        grad_b = (2 / n) * sai_so.sum()

        w = w - lr * grad_w                      # 4. cập nhật
        b = b - lr * grad_b

    return w, b, lich_su


w, b, lich_su = train(X, y)

print(f"w học được : {w.round(4)}      (thật: {W_THAT})")
print(f"b học được : {b:.4f}            (thật: {B_THAT})")
print(f"\nloss đầu   : {lich_su[0]:.4f}")
print(f"loss cuối  : {lich_su[-1]:.4f}   <- xấp xỉ mức nhiễu 0.3² = 0.09")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2)

ax1.plot(lich_su, lw=2)
ax1.set_xlabel("epoch"); ax1.set_ylabel("MSE"); ax1.set_title("Loss theo epoch")
ax1.grid(alpha=.3)

ax2.plot(lich_su, lw=2); ax2.set_yscale("log")
ax2.set_xlabel("epoch"); ax2.set_title("Cùng dữ liệu, trục y thang log")
ax2.grid(alpha=.3)

plt.tight_layout(); plt.show()

> 💡 **Mẹo thực chiến:** khi loss trông "phẳng lì", hãy vẽ lại với `yscale("log")`. Rất nhiều lúc model vẫn đang tiến bộ, chỉ là thang tuyến tính không cho thấy.

### So sánh nhiều learning rate trên bài toán thật

In [ ]:
for lr in [0.001, 0.01, 0.1, 0.5, 1.0]:
    _, _, ls = train(X, y, lr=lr, so_epoch=300)
    ket_thuc = ls[-1]
    nhan = f"{ket_thuc:.4f}" if np.isfinite(ket_thuc) and ket_thuc < 1e6 else "PHÂN KỲ"
    plt.plot(ls[:100], label=f"lr={lr} (cuối: {nhan})", lw=2)

plt.ylim(0, 40); plt.xlabel("epoch"); plt.ylabel("MSE")
plt.legend(); plt.grid(alpha=.3); plt.title("Ảnh hưởng của learning rate")
plt.show()

## 8. Bốn bước bạn sẽ lặp lại suốt phần đời còn lại

```python
y_pred = model(X)              # 1. Forward  — dự đoán
loss   = ham_loss(y_pred, y)   # 2. Loss     — sai bao nhiêu
grads  = tinh_gradient(loss)   # 3. Backward — sai theo hướng nào
params = params - lr * grads   # 4. Update   — sửa lại
```

Ở Phase 05, PyTorch sẽ viết hộ bạn bước 3 (đó là **autograd**), nhưng bốn bước này giữ nguyên **y hệt**:

```python
y_pred = model(X)
loss = criterion(y_pred, y)
loss.backward()        # PyTorch tự tính gradient
optimizer.step()       # PyTorch tự cập nhật
```

Bạn vừa hiểu chính xác chuyện gì xảy ra bên trong hai dòng đó.

## ✍️ Bài tập

```bash
pytest tests/phase02/test_ex02.py -v
```